# Week 3 Session 2: EDA: The Story the Titanic Data Tells
**Forge Academy | Data Science Foundations**

**By the end of this session you can:**
1. Explain what EDA is and follow a simple workflow: question, summarize, visualize, interpret, caveat.
2. Describe one column at a time with statistics and charts (histograms, bar charts, boxplots).
3. Compare survival across groups with `groupby`, `crosstab` and `pivot_table`.
4. Look at several variables together with a heatmap and a correlation matrix, and know why correlation is not causation.
5. Turn your findings into a short, honest story with clear charts.

**Setup:** keep `Titanic-Dataset.csv` in the same folder as this notebook. Open it in VS Code, select your Python kernel, and run cells from top to bottom. We use pandas and matplotlib. If `import matplotlib` fails, run `pip install matplotlib` in the terminal.

## Today's mission

Last session the Lagos Heritage Museum asked us for **a clean, trustworthy passenger list**. We delivered it.

Now they are back with a bigger request:

**"We have the clean data. Now tell us the story it tells. Who survived, who did not, and why might that be?"**

That is Exploratory Data Analysis (EDA). Every section today answers one museum question:

| Section | Museum question |
|---|---|
| 0 | Can we rebuild the clean list quickly? |
| 1 | How do analysts explore data? |
| 2 | Who was on board? (one column at a time) |
| 3 | Who survived? (survival against one factor) |
| 4 | How do factors combine? |
| 5 | What story do we tell the visitors? |
| 6 | Challenge, recap and assignment |

## 0. Rebuild the clean passenger list

We cleaned the data step by step last session. Here is the same cleaning again, one short step per cell, so we start from trustworthy data:
- Embarked: 2 missing, filled with the most common port.
- Title extracted from Name and rare titles grouped.
- Age: 177 missing, filled with the median age of each title.
- Cabin: 77% missing, kept only as a HasCabin flag, then dropped.
- New columns: FamilySize and Alone.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
raw = pd.read_csv("Titanic-Dataset.csv")
df = raw.copy()

In [ ]:
df.head() #preview

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0]) #filling the missing values with the most common value, cos in this case it's the most likely value

In [5]:
df["Title"] = df["Name"].str.extract(r", ([A-Za-z ]+)\.")[0].str.strip() #extracting the title from the name, and stripping any whitespace

In [6]:
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}) #replacing some titles with more common ones

In [7]:
df.loc[~df["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Title"] = "Rare" #replacing any titles that are not in the list with "Rare"

In [ ]:
df["Age"] = df["Age"].fillna(df.groupby("Title")["Age"].transform("median")) 

In [ ]:
df["HasCabin"] = df["Cabin"].notnull()
df = df.drop(columns=["Cabin"])

In [ ]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["Alone"] = df["FamilySize"] == 1

In [ ]:
df.shape

In [ ]:
df.isnull().sum().sum()

In [ ]:
df.head()

### 0.1 A first look at the new Title column
**Museum question: What kinds of people were on board?**

Titles like Mr, Mrs, Miss and Master tell us sex and rough age in one word. Let us count them.

In [ ]:
df["Title"].value_counts()

Now the median age for each title. This is exactly what we used to fill the missing ages.

In [ ]:
df.groupby("Title")["Age"].median()

Reading it: "Master" was used for boys, so its median age is very low. That is why filling ages by title is better than one overall median.

## 1. What is EDA?

**Exploratory Data Analysis** means getting to know a dataset before you build anything or make claims. You look for patterns, surprises and problems.

We follow the same five steps for every question today:

| Step | What you do | Example |
|---|---|---|
| 1. Question | Ask one clear question | "Did women survive more than men?" |
| 2. Summarize | Get the numbers | `groupby("Sex")["Survived"].mean()` |
| 3. Visualize | Draw a chart that shows it | Bar chart of survival rate by sex |
| 4. Interpret | Say what it means in one sentence | "Women survived about 4 times as often." |
| 5. Caveat | Say what could mislead | "This is a pattern, not proof of cause." |

**Two kinds of variables**
- **Numeric** (numbers you can average): Age, Fare, FamilySize. Use histograms and boxplots.
- **Categorical** (labels or groups): Sex, Pclass, Embarked, Title. Use counts and bar charts.

`Pclass` is stored as 1, 2, 3 but it is a category: averaging ticket classes makes no sense.

## 2. Who was on board? One column at a time (univariate analysis)

### 2.1 Describing numbers
**Museum question: How old were the passengers, and what did they pay?**

| Statistic | Meaning |
|---|---|
| mean | the average; pulled by extreme values |
| median | the middle value; not pulled by extremes |
| mode | the most common value |
| std | standard deviation: how spread out values are around the mean |
| quantiles | 25% (Q1), 50% (median), 75% (Q3) cut points |

In [ ]:
df[["Age", "Fare"]].describe().round(2)

In [ ]:
print("Age  mean:", round(df["Age"].mean(), 1), " median:", df["Age"].median(), " mode:", df["Age"].mode()[0])

In [ ]:
print("Fare mean:", round(df["Fare"].mean(), 2), " median:", df["Fare"].median())

**A tiny example to see why.** Five tickets: four ordinary ones and one very expensive one.

In [ ]:
tickets = pd.Series([10, 12, 14, 15, 500])
print("mean:", tickets.mean())
print("median:", tickets.median())

One extreme ticket drags the mean up to 110.2, but the median stays at 14, which looks like a normal ticket. Keep this picture in mind for the real fares below.

Notice: Fare's mean (32.2) is more than double its median (14.45). That tells us a few very expensive tickets are pulling the average up. For Fare, the median is the better "typical" value.

### 2.2 Histograms: the shape of a numeric column
A **histogram** splits values into ranges (bins) and counts how many fall in each. Tall bars = many passengers in that range.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["Age"], bins=30, color="steelblue", edgecolor="white")
plt.title("Most passengers were young adults (20 to 40)")
plt.xlabel("Age (years)")
plt.ylabel("Number of passengers")
plt.show()

Every chart has a **title that states the takeaway**, and labelled axes. We will keep that habit all session.

### Your turn 1
Draw a histogram of **Fare** with 40 bins. Give it a takeaway title and axis labels. What shape do you see?

In [ ]:
# Your code here


### 2.3 Skew and the log trick
**Skew** measures how lopsided a distribution is. 0 = balanced. Above 1 = a long tail to the right.

A common fix for heavy right skew is to look at the **log** of the values. Log squashes big numbers more than small ones, so the shape becomes easier to see. We use `np.log1p`, which is log(1 + x), so a fare of 0 does not break it.

In [ ]:
import numpy as np

In [ ]:
print("Age skew:", round(df["Age"].skew(), 2))

In [ ]:
print("Fare skew:", round(df["Fare"].skew(), 2))

In [ ]:
df["LogFare"] = np.log1p(df["Fare"])
print("LogFare skew:", round(df["LogFare"].skew(), 2))

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["LogFare"], bins=30, color="seagreen", edgecolor="white")
plt.title("On a log scale, fares look much more balanced")
plt.xlabel("log(1 + Fare)")
plt.ylabel("Number of passengers")
plt.show()

### 2.4 Outliers: IQR and boxplots
**Museum question: Were some fares unusually high?**

The **IQR** (interquartile range) is Q3 minus Q1: the range of the middle 50% of values. A common rule: a value is an **outlier** if it is above Q3 + 1.5 x IQR or below Q1 - 1.5 x IQR.

In [ ]:
q1 = df["Fare"].quantile(0.25)
q3 = df["Fare"].quantile(0.75)
print("Q1:", round(q1, 2), " Q3:", round(q3, 2))

In [ ]:
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print("IQR:", round(iqr, 2), " Upper fence:", round(upper, 2))

In [ ]:
print("Fares above the fence:", (df["Fare"] > upper).sum())

In [ ]:
plt.figure(figsize=(8, 3))
plt.boxplot(df["Fare"], vert=False)
plt.title("Fare: a box of typical fares and a long line of high outliers")
plt.xlabel("Fare (pounds)")
plt.show()

**How to read a boxplot:** the box spans Q1 to Q3, the line inside is the median, the whiskers reach the furthest normal values, and dots are outliers.

**Outlier does not mean error.** These high fares are real first class tickets. We keep them, but we remember they pull the mean up.

### Your turn 2
Compute Q1, Q3, IQR and the upper fence for **Age**. How many ages are outliers? Then draw a boxplot of Age.

In [ ]:
# Your code here


### 2.5 Counting categories
**Museum question: Who travelled? How many men and women, in which class, from which port?**

For categories we count with `value_counts()` and draw a **bar chart**. pandas can plot directly with `.plot(kind="bar")`.

In [ ]:
df["Sex"].value_counts()

In [ ]:
df["Sex"].value_counts(normalize=True).round(3)

In [ ]:
df["Sex"].value_counts().plot(kind="bar", color=["steelblue", "salmon"], figsize=(5, 4))
plt.title("Almost two thirds of passengers were men")
plt.xlabel("Sex")
plt.ylabel("Number of passengers")
plt.xticks(rotation=0)
plt.show()

### Your turn 3
Make count bar charts for **Pclass** and **Embarked**. Use `sort_index()` for Pclass so classes appear in order 1, 2, 3. Add takeaway titles.

In [ ]:
# Your code here


In [ ]:
# Your code here


## 3. Who survived? Survival against one factor (bivariate analysis)

Because `Survived` is 0 or 1, **its mean is the survival rate**. Example: values 1, 0, 0, 1 have mean 0.5, which is 50% survived.

In [ ]:
print("Overall survival rate:", round(df["Survived"].mean() * 100, 1), "%")

In [ ]:
df["Survived"].value_counts()

### 3.1 Survival by sex
**Museum question: Did "women and children first" really happen?**

In [ ]:
by_sex = df.groupby("Sex")["Survived"].mean()
by_sex.round(3)

In [ ]:
by_sex.plot(kind="bar", color=["salmon", "steelblue"], figsize=(5, 4))
plt.title("Women survived about 4 times as often as men")
plt.ylabel("Survival rate")
plt.xlabel("Sex")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()

**Interpret:** about 74% of women survived against 19% of men. **Caveat:** sex is linked to other things (class, travelling with children), so this alone does not tell us everything.

### 3.2 Survival by class, with `agg`
`agg` gives several numbers at once. Always show the **count** next to a rate: a rate from 7 people is far less reliable than one from 500.

In [ ]:
by_class = df.groupby("Pclass")["Survived"].agg(["count", "sum", "mean"])
by_class.columns = ["passengers", "survivors", "survival_rate"]
by_class.round(3)

In [ ]:
by_class["survival_rate"].plot(kind="bar", color="slategray", figsize=(5, 4))
plt.title("First class passengers were 2.6 times as likely to survive as third")
plt.ylabel("Survival rate")
plt.xlabel("Ticket class")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()

### 3.2b Compare with the baseline
A rate means more next to the overall 38.4%. Subtracting the overall rate shows how far above or below average each class was.

In [ ]:
overall = df["Survived"].mean()
(by_class["survival_rate"] - overall).round(3)

Positive means better than average, negative means worse. Third class was about 14 points below average.

### 3.2c Survival by cabin record
**Museum question: Did having a recorded cabin go with survival?** We kept `HasCabin` from last session's cleaning.

In [ ]:
df.groupby("HasCabin")["Survived"].agg(["count", "mean"]).round(3)

In [ ]:
pd.crosstab(df["HasCabin"], df["Pclass"])

**Interpret:** passengers with a recorded cabin survived far more often. **Caveat:** most recorded cabins belong to first class passengers, so this is largely the class story again, not the cabin itself.

### 3.3 `crosstab`: counts in a grid
`pd.crosstab` counts how often each combination happens. `normalize="index"` turns each row into percentages.

In [ ]:
pd.crosstab(df["Pclass"], df["Survived"])

In [ ]:
pd.crosstab(df["Pclass"], df["Survived"], normalize="index").round(3)

### 3.4 `pivot_table`: the same idea, more flexible
`pivot_table` groups rows by `index` and applies `aggfunc` (mean by default) to `values`.

In [ ]:
df.pivot_table(index="Embarked", values="Survived", aggfunc=["count", "mean"]).round(3)

### 3.5 Survival by age group
`pd.cut` sorts a number column into labelled ranges (bins).

In [ ]:
bins = [0, 12, 18, 35, 60, 100]
labels = ["Child 0-12", "Teen 13-18", "Young adult 19-35", "Adult 36-60", "Senior 61+"]
df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels)
df[["Age", "AgeGroup"]].head()

In [ ]:
by_age = df.groupby("AgeGroup", observed=True)["Survived"].agg(["count", "mean"]).round(3)
by_age

In [ ]:
by_age["mean"].plot(kind="bar", color="mediumpurple", figsize=(7, 4))
plt.title("Children had the best survival rate; seniors the worst")
plt.ylabel("Survival rate")
plt.xlabel("Age group")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.show()

### Your turn 4: family
**Museum question: Did travelling with family help?**
1. Survival rate and count by `Alone`.
2. Survival rate and count by `FamilySize`. Draw a bar chart of the rate.
3. Write one sentence interpreting it, and one caveat.

In [ ]:
# Your code here


In [ ]:
# Your code here


**Interpretation and caveat:** (write yours here)

### Your turn 5: fare bands
`pd.qcut(df["Fare"], 4, labels=[...])` splits fares into 4 equal-sized groups (quartiles). Create `FareBand` with labels `"Q1 lowest", "Q2", "Q3", "Q4 highest"`, then find and plot the survival rate per band.

In [ ]:
# Your code here


## 4. How do factors combine? (multivariate analysis)

### 4.1 Sex and class together
**Museum question: Was it sex or class that mattered? Or both?**

In [ ]:
sex_class = df.pivot_table(index="Sex", columns="Pclass", values="Survived", aggfunc="mean").round(3)
sex_class

A **heatmap** colours each cell of a table by its value. With matplotlib we use `imshow` and write the numbers on top.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
im = ax.imshow(sex_class.values, cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks(range(len(sex_class.columns)))
ax.set_xticklabels(["1st", "2nd", "3rd"])
ax.set_yticks(range(len(sex_class.index)))
ax.set_yticklabels(sex_class.index)
for i in range(sex_class.shape[0]):
    for j in range(sex_class.shape[1]):
        ax.text(j, i, round(sex_class.values[i, j], 2), ha="center", va="center")
ax.set_title("Survival rate by sex and class: both mattered")
ax.set_xlabel("Ticket class")
fig.colorbar(im, ax=ax, label="Survival rate")
plt.show()

### 4.1b Women and children first?
We can group by two columns at once. `unstack()` turns the second group into columns so it reads like a table.

In [ ]:
sex_age = df.groupby(["Sex", "AgeGroup"], observed=True)["Survived"].mean().round(3)
sex_age

In [ ]:
sex_age.unstack()

Boys up to 12 survived at a similar rate to girls, but from 13 onwards the male rate drops sharply. That is "women and children first" showing in the data.

### Your turn 6: test the Cherbourg question
Make a pivot table of survival rate with `Embarked` as rows and `Pclass` as columns. Also make a `crosstab` of Embarked vs Pclass with `normalize="index"` to see the class mix at each port. Was Cherbourg itself lucky, or did it just have more first class passengers?

In [ ]:
# Your code here


### 4.2 Age by survival
**Museum question: Did age shape survival?** We overlay two histograms: survivors and non-survivors. `alpha` makes them see-through. `density=True` scales each to the same total area so groups of different sizes compare fairly.

In [ ]:
survived = df[df["Survived"] == 1]["Age"]
died = df[df["Survived"] == 0]["Age"]

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(died, bins=30, alpha=0.5, density=True, label="Did not survive", color="firebrick")
plt.hist(survived, bins=30, alpha=0.5, density=True, label="Survived", color="seagreen")
plt.title("Young children survived more; the age shapes are otherwise similar")
plt.xlabel("Age (years)")
plt.ylabel("Share of group")
plt.legend()
plt.show()

In [ ]:
print("Median age, survived:", survived.median(), " did not survive:", died.median())

### Your turn 7
Make a boxplot comparing Fare for survivors and non-survivors. Hint: `plt.boxplot([died_fares, survived_fares], labels=[...])`. Use `showfliers=False` to hide outliers so the boxes are readable.

In [ ]:
# Your code here


### 4.3 Correlation matrix
**Correlation** (from -1 to 1) measures how strongly two numeric columns move together in a straight line.
- Near +1: when one goes up, the other goes up.
- Near -1: when one goes up, the other goes down.
- Near 0: no straight-line relationship (there may still be a curved one).

In [ ]:
num_cols = ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize"]
corr = df[num_cols].corr().round(2)
corr

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45)
ax.set_yticks(range(len(num_cols)))
ax.set_yticklabels(num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        ax.text(j, i, corr.values[i, j], ha="center", va="center", fontsize=8)
ax.set_title("Correlation between numeric columns")
fig.colorbar(im, ax=ax)
plt.show()

**Correlation is not causation.** Fare and survival are positively correlated, but paying more did not *cause* survival. Richer passengers had cabins nearer the lifeboats. A hidden third factor (here, class and cabin location) can create a correlation.

### Your turn 8
Add a numeric column `IsFemale` (1 for female, 0 for male) using `map`, and find its correlation with `Survived`. How does it compare with Pclass and Fare?

In [ ]:
# Your code here


## 5. Telling the story

**Museum question: What do we put on the exhibition wall?**

Analysis only matters if people understand it. A good finding has: **a number, a comparison, and a plain sentence.**

### Our 5 key findings (from today's numbers)
1. **Most did not survive.** About 38% of the 891 passengers in this data survived.
2. **Sex was the strongest factor.** About 74% of women survived against 19% of men.
3. **Class mattered a lot.** First class 63%, second 47%, third 24%.
4. **Both together tell the real story.** Nearly all first and second class women survived (97% and 92%); third class women had a 50% chance; a first class man (37%) did worse than a third class woman.
5. **Children and small families did better.** Children up to 12 survived at 58%; families of 2 to 4 survived at 55 to 72%, against 30% for people travelling alone.

### Limitations (always say these)
- **Missing ages were estimated.** 177 ages (20%) are title medians, which flattens age patterns.
- **This is not everyone.** The ship carried about 2,224 people; this dataset has 891 passengers (no crew).
- **Survivorship in the records:** information like cabins was better recorded for some groups (mostly first class), so missing data is not random.
- **Patterns, not proof.** We found associations; the causes (deck location, lifeboat rules, crew decisions) come from history, not this table.

### How to communicate
- **Chart titles state the takeaway** ("Women survived 4 times as often"), not just the topic ("Survival by sex").
- **Label axes** with units; start rate axes at 0.
- **One sentence under each chart:** what it shows and why it matters.
- **Show counts** next to rates when groups are small.

### Your turn 9
Pick ONE chart from today and write a museum wall caption for it: a takeaway title, one sentence with a number, and one caveat.

**My caption:** (write yours here)

## 6. Challenge (5 to 10 minutes)
**Museum question: Who were the most and least likely to survive?**

Make a pivot table of survival rate with `Title` as rows and `Pclass` as columns, plus a count table using `aggfunc="count"`. Which title and class combination had the highest survival rate, and which the lowest (only consider groups with at least 10 people)? Write two sentences.

In [ ]:
# Your code here


## Recap
- EDA follows five steps: question, summarize, visualize, interpret, caveat.
- One column at a time: `describe`, mean vs median, histograms, skew, IQR and boxplots, `value_counts` bar charts.
- Survival against a factor: `groupby(...)["Survived"].agg(["count", "mean"])`, `crosstab`, `pivot_table`, `cut` and `qcut`.
- Factors combine: the Sex x Class heatmap tells a truer story than either alone; correlation is not causation, and confounders (like class behind port) can mislead.
- A finding needs a number, a comparison, a plain sentence and a caveat; chart titles state the takeaway.

## Assignment: your own Titanic EDA
**Due:** before our next class. **Submit:** your `.ipynb` file and your write-up through the usual submission form.

**Task**
1. A clean notebook that starts from `Titanic-Dataset.csv`, cleans it (justify each fix), and explores it with the five-step workflow.
2. **3 to 5 charts**, each with a takeaway title, labelled axes and a one-sentence interpretation. At least one must combine two factors (like Sex x Class).
3. A short **written story** (300 to 600 words): written in a markdown cell at the end of your notebook or a separate document, using your charts, **3 to 5 findings with numbers**, and at least 2 limitations.

**Rubric (20 points, 0 to 4 each)**
| Area | What earns full marks |
|---|---|
| Correctness | Code runs top to bottom; numbers and charts are right |
| Problem solving | Clear questions; goes beyond single-factor charts |
| Code quality | Clear names, no repeated code, raw data untouched |
| Communication | Takeaway titles, labels, readable story, honest caveats |
| Independence | Your own questions and words; you can explain every line |

AI tools are allowed, but you must be able to explain every line and every chart on our call.